In [104]:
import pandas as pd
import numpy as np

# Extract and inspect
# Read the raw CSV into a dataframe

df = pd.read_csv(r"C:\Users\Donnchadh\Desktop\Portfolio\sale_data_pipeline\data\raw\dirty_cafe_sales.csv")

# Basic data inspection
# everything comes in as a string by default and use value_counts alongside isnull()

print(df.shape)
print(df.columns)
print(df.dtypes)
print(df.head())
print(df.isnull().sum())
for col in df.columns:
    print(f"\n{col}:\n{df[col].value_counts(dropna=False)}")

(10000, 8)
Index(['Transaction ID', 'Item', 'Quantity', 'Price Per Unit', 'Total Spent',
       'Payment Method', 'Location', 'Transaction Date'],
      dtype='str')
Transaction ID      str
Item                str
Quantity            str
Price Per Unit      str
Total Spent         str
Payment Method      str
Location            str
Transaction Date    str
dtype: object
  Transaction ID    Item Quantity Price Per Unit Total Spent  Payment Method  \
0    TXN_1961373  Coffee        2            2.0         4.0     Credit Card   
1    TXN_4977031    Cake        4            3.0        12.0            Cash   
2    TXN_4271903  Cookie        4            1.0       ERROR     Credit Card   
3    TXN_7034554   Salad        2            5.0        10.0         UNKNOWN   
4    TXN_3160411  Coffee        2            2.0         4.0  Digital Wallet   

   Location Transaction Date  
0  Takeaway       2023-09-08  
1  In-store       2023-05-16  
2  In-store       2023-07-19  
3   UNKNOWN       2023-

In [114]:
# Transform Item column

# Inspecting unique values for cross referencing 
print(df['Item'].unique())
print(df['Price Per Unit'].unique())
print(df[['Item', 'Price Per Unit']].drop_duplicates().sort_values('Price Per Unit'))

# Replace ERROR and UNKNOWN with NaN
# Fill where Price Per Unit maps to a single unique item
# 3.0 and 4.0 map to multiple items so cannot be filled confidently

df['Item'] = df['Item'].replace({'ERROR': np.nan, 'UNKNOWN': np.nan})
df['Item'] = df['Item'].fillna(df['Price Per Unit'].astype(str).map({'1.0': 'Cookie',
                                                                     '1.5': 'Tea',
                                                                     '2.0': 'Coffee',
                                                                     '5.0': 'Salad'}))
# Flag results for manual check
df['Item_Check'] = df['Item'].isnull()

# Validating results
df['Item'].isnull()
print(f" Null values in Item: {df['Item'].isnull().sum()}")
print(f"Rows with mismatched Item: {(df['Item'] != 0).sum()}")

<StringArray>
[  'Coffee',     'Cake',   'Cookie',    'Salad', 'Smoothie',        nan,
 'Sandwich',      'Tea',    'Juice']
Length: 9, dtype: str
[2.0 3.0 1.0 5.0 4.0 1.5 nan]
         Item Price Per Unit
2      Cookie            1.0
14        Tea            1.5
0      Coffee            2.0
1        Cake            3.0
6         NaN            3.0
17      Juice            3.0
5    Smoothie            4.0
7    Sandwich            4.0
165       NaN            4.0
3       Salad            5.0
56       Cake            NaN
65   Sandwich            NaN
104     Juice            NaN
118       NaN            NaN
186  Smoothie            NaN
 Null values in Item: 501
Rows with mismatched Item: 10000


In [113]:
# Transform Price Per Unit column 
# Replace ERROR and UNKNOWN with NaN
# Fill where Price Per Unit maps to a single unique item, reverse of previous step

df['Price Per Unit'] = df['Price Per Unit'].replace({'ERROR': np.nan, 'UNKNOWN': np.nan})
df['Price Per Unit'] = df['Price Per Unit'].fillna(df['Item'].astype(str).map({'Cookie': '1.0',
                                                                               'Tea': '1.50',
                                                                               'Coffee': '2.0',
                                                                               'Salad': '5.0' }))
# Round for clean output
df['Price Per Unit'] = df['Price Per Unit'].round(2)

# Flag results for manual check
df['Price_Per_Unit_Check'] = df['Price Per Unit'].isnull()

# Validate results
df['Price_Per_Unit_Check'] = df['Price Per Unit'].isnull()
print(f"Price Per Unit null count: {df['Price Per Unit'].isnull().sum()}")
print(f"Rows with mismatched Price Per Unit: {(df['Price Per Unit'] != 0).sum()}")

Price Per Unit null count: 278
Rows with mismatched Price Per Unit: 10000


In [112]:
# Transform Quantity column
# # Convert columns to numeric - pd.to_numeric handles text values like ERROR by converting them to null instead of crashing unlike astype() as I found out

df['Total Spent'] = pd.to_numeric(df['Total Spent'], errors='coerce')
df['Price Per Unit'] = pd.to_numeric(df['Price Per Unit'], errors='coerce')
df['Quantity'] = pd.to_numeric(df['Quantity'], errors='coerce')

# Fill missing Quantity by dividing Total Spent by Price Per Unit while themask limits the calculation to rows where both values exist

mask = df[['Total Spent', 'Price Per Unit']].notnull().all(axis=1)
cal_quant = df['Total Spent'] / df['Price Per Unit']
df['Quantity'] = df['Quantity'].fillna(cal_quant[mask])

# Round results for cleaner output
df['Quantity'] = df['Quantity'].round(0)

# Flag results for manual check
df['Quantity_Check'] = df['Quantity'].isnull()

# Validate outputs
df['Quantity_check'] = df['Quantity'].isnull()
print(f"Null values in Quantity: {df['Quantity'].isnull().sum()}")
print(f"Rows with mismatched Quantity: {(df['Quantity'] != 0).sum()}")

Null values in Quantity: 29
Rows with mismatched Quantity: 10000


In [111]:
# Transform Total Spent column

# Extract missing Total Spent by multiplying Quantity by Price Per Unit
# Cross reference existing values against calculation

mask = df[['Quantity', 'Price Per Unit']].notnull().all(axis=1)
cal_total_spent = df['Quantity'] * df['Price Per Unit']
df['Total Spent'] = df['Total Spent'].fillna(cal_total_spent[mask])
df['Total Spent'] = df['Total Spent'].round(2)

# Flag results for manual check
df['Total_Spent_Check'] = df['Total Spent'] - (df['Quantity'] * df['Price Per Unit'])

# Validate results
df['Total Spent Check'] = df['Total Spent'].notnull().sum()

print(f"Null values in Total Spent: {df['Total Spent'].isnull().sum()}")
print(f"Rows with mismatched Total Spent: {(df['Total_Spent_Check'] != 0).sum()}")

Null values in Total Spent: 32
Rows with mismatched Total Spent: 298


In [109]:
# Transform - Flag remaining nulls for review
# Transaction Id, Payment Method, Location, and Transaction Date need to be manually checked against the source system for null values
# Created a review dataframe containaing remaining null values

df['Payment_Method_Check'] = df['Payment Method'].isnull()
df['Location_Check'] = df['Location'].isnull()
df['Transaction_Date_Check'] = df['Transaction Date'].isnull()

check_cols = [
    'Item_Check',
    'Price_Per_Unit_Check',
    'Quantity_Check',
    'Total_Spent_Check',
    'Payment_Method_Check',
    'Location_Check',
    'Transaction_Date_Check'
]

df_review = df[df[check_cols].any(axis=1)]
print(f"Rows flagged for manual review: {df_review.shape[0]}")

Rows flagged for manual review: 5435


In [110]:
# Load 

# Clean df is original df with all check columns dropped - no rows removed
# Review df contains all flagged rows for manual investigation
# Both loaded to CSV

df_clean = df.drop(columns=check_cols)

df_clean.to_csv(
    r"C:\Users\Donnchadh\Desktop\Portfolio\sale_data_pipeline\data\clean\cafe_sales_clean.csv",
    index=False
)

df_review.to_csv(
    r"C:\Users\Donnchadh\Desktop\Portfolio\sale_data_pipeline\data\clean\cafe_sales_review.csv",
    index=False
)

print(f"Clean rows saved: {df_clean.shape[0]}")
print(f"Review rows saved: {df_review.shape[0]}")

Clean rows saved: 10000
Review rows saved: 5435
